# 317 — Layer 3 Assessment Cleanup

Removes **Assessment**, **Finding**, and **ReasoningStep** nodes (and all their
relationships) written to Neo4j by the agent pipeline.

Run this notebook to reset Layer 3 back to a clean state without touching
Layer 1 (entities) or Layer 2 (regulatory graph).

In [ ]:
%run 311_agent_setup.ipynb

## 1. Inspect current Layer 3 state

In [ ]:
# Count Layer 3 nodes before cleanup
counts = conn.run_query("""
    MATCH (a:Assessment)
    OPTIONAL MATCH (a)-[:HAS_FINDING]->(f:Finding)
    OPTIONAL MATCH (a)-[:HAS_STEP]->(rs:ReasoningStep)
    RETURN
        count(DISTINCT a)  AS assessments,
        count(DISTINCT f)  AS findings,
        count(DISTINCT rs) AS reasoning_steps
""")
print('Layer 3 nodes before cleanup:')
for row in counts:
    print(f'  Assessments:     {row["assessments"]}')
    print(f'  Findings:        {row["findings"]}')
    print(f'  ReasoningSteps:  {row["reasoning_steps"]}')

## 2. Preview assessments (optional)

In [ ]:
# List assessments that will be deleted
rows = conn.run_query("""
    MATCH (a:Assessment)
    RETURN a.assessment_id AS id,
           a.entity_id     AS entity,
           a.verdict       AS verdict,
           a.agent         AS agent,
           a.created_at    AS created_at
    ORDER BY a.created_at DESC
    LIMIT 50
""")
print(f'{len(rows)} assessment(s) found:')
for r in rows:
    print(f'  {r["id"]}  entity={r["entity"]}  verdict={r["verdict"]}  agent={r["agent"]}')

## 3. Delete Layer 3 nodes

> **Warning** — this is irreversible. Run the preview cell above first.

In [ ]:
result = conn.run_query("""
    MATCH (a:Assessment)
    OPTIONAL MATCH (a)-[:HAS_FINDING]->(f:Finding)
    OPTIONAL MATCH (a)-[:HAS_STEP]->(rs:ReasoningStep)
    DETACH DELETE a, f, rs
""")
print('Layer 3 nodes deleted.')

## 4. Verify cleanup

In [ ]:
# Confirm all Layer 3 nodes are gone
after = conn.run_query("""
    MATCH (a:Assessment)
    OPTIONAL MATCH (a)-[:HAS_FINDING]->(f:Finding)
    OPTIONAL MATCH (a)-[:HAS_STEP]->(rs:ReasoningStep)
    RETURN
        count(DISTINCT a)  AS assessments,
        count(DISTINCT f)  AS findings,
        count(DISTINCT rs) AS reasoning_steps
""")
row = after[0]
assert row['assessments'] == 0, 'Assessments still present!'
assert row['findings'] == 0, 'Findings still present!'
assert row['reasoning_steps'] == 0, 'ReasoningSteps still present!'
print('All Layer 3 nodes removed. Graph is clean.')